# SOH Boundary And Outlier Masking

This notebook tests whether observed SOH spikes can be reduced by:
- trimming the start and end of each event
- removing rows with nonphysical or outlier telemetry
- recomputing event-level SOH after masking

The goal is to see whether the spikes are driven by unstable estimator periods rather than true SOH changes.

In [1]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)


def find_path(*relative_candidates: str) -> Path:
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        for relative in relative_candidates:
            candidate = (root / relative).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Could not find any of: {relative_candidates}')


timeseries_path = find_path('data/event_timeseries.parquet', 'data/processed/event_timeseries.parquet')
PLANES = ['166', '192']
BOUNDARY_TRIM_MIN = 2.0
SOH_DELTA_THRESHOLD = 5.0
SOC_DELTA_THRESHOLD = 3.0
VOLT_DELTA_THRESHOLD = 5.0
CELL_SPREAD_ABS_THRESHOLD = 0.25
TEMP_JUMP_THRESHOLD = 3.0

timeseries_path


PosixPath('/Users/benfogerty/Desktop/EPlaneCapstone/CapstoneEPlane/data/event_timeseries.parquet')

In [ ]:
def robust_zscore(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce')
    med = s.median(skipna=True)
    mad = (s - med).abs().median(skipna=True)
    if pd.isna(mad) or float(mad) <= 1e-12:
        return pd.Series(np.nan, index=s.index)
    return 0.6745 * (s - med) / mad


def load_sample_level(plane_id: str) -> pd.DataFrame:
    cols = [
        'plane_id', 'flight_id', 'event_datetime', ' time(min)',
        ' bat 1 soh', ' bat 2 soh',
        ' bat 1 current', ' bat 2 current',
        ' bat 1 voltage', ' bat 2 voltage',
        ' bat 1 soc', ' bat 2 soc',
        ' bat 1 avg cell temp', ' bat 2 avg cell temp',
        ' bat 1 min cell volt', ' bat 2 min cell volt',
        ' bat 1 max cell volt', ' bat 2 max cell volt',
    ]
    df = pd.read_parquet(timeseries_path, columns=cols, filters=[('plane_id', '==', plane_id)]).copy()
    df['event_datetime'] = pd.to_datetime(df['event_datetime'])
    df = df.rename(columns={
        ' time(min)': 'time_min',
        ' bat 1 soh': 'battery_1_soh',
        ' bat 2 soh': 'battery_2_soh',
        ' bat 1 current': 'battery_1_current_a',
        ' bat 2 current': 'battery_2_current_a',
        ' bat 1 voltage': 'battery_1_voltage_v',
        ' bat 2 voltage': 'battery_2_voltage_v',
        ' bat 1 soc': 'battery_1_soc_pct',
        ' bat 2 soc': 'battery_2_soc_pct',
        ' bat 1 avg cell temp': 'battery_1_temp_c',
        ' bat 2 avg cell temp': 'battery_2_temp_c',
        ' bat 1 min cell volt': 'battery_1_cell_v_min',
        ' bat 2 min cell volt': 'battery_2_cell_v_min',
        ' bat 1 max cell volt': 'battery_1_cell_v_max',
        ' bat 2 max cell volt': 'battery_2_cell_v_max',
    })

    numeric_cols = [c for c in df.columns if c not in ['plane_id', 'event_datetime']]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    zero_placeholder_cols = [
        'battery_1_soh', 'battery_2_soh',
        'battery_1_voltage_v', 'battery_2_voltage_v',
        'battery_1_soc_pct', 'battery_2_soc_pct',
        'battery_1_temp_c', 'battery_2_temp_c',
        'battery_1_cell_v_min', 'battery_2_cell_v_min',
        'battery_1_cell_v_max', 'battery_2_cell_v_max',
    ]
    df[zero_placeholder_cols] = df[zero_placeholder_cols].replace(0, pd.NA)

    parts = []
    for battery_id in [1, 2]:
        out = df[['plane_id', 'flight_id', 'event_datetime', 'time_min']].copy()
        out['battery_id'] = battery_id
        out['observed_soh_pct'] = df[f'battery_{battery_id}_soh']
        out['current_a'] = df[f'battery_{battery_id}_current_a']
        out['voltage_v'] = df[f'battery_{battery_id}_voltage_v']
        out['soc_pct'] = df[f'battery_{battery_id}_soc_pct']
        out['temp_c'] = df[f'battery_{battery_id}_temp_c']
        out['cell_v_min'] = df[f'battery_{battery_id}_cell_v_min']
        out['cell_v_max'] = df[f'battery_{battery_id}_cell_v_max']
        out['cell_v_spread'] = out['cell_v_max'] - out['cell_v_min']
        parts.append(out)

    long_df = pd.concat(parts, ignore_index=True)
    long_df = long_df.dropna(subset=['observed_soh_pct']).sort_values(['plane_id', 'battery_id', 'flight_id', 'time_min']).reset_index(drop=True)
    return long_df


sample_df = pd.concat([load_sample_level(plane) for plane in PLANES], ignore_index=True)
sample_df.head()


In [ ]:
sample_df['event_duration_min'] = sample_df.groupby(['plane_id', 'battery_id', 'flight_id'])['time_min'].transform('max')
sample_df['boundary_mask'] = (sample_df['time_min'] < BOUNDARY_TRIM_MIN) | ((sample_df['event_duration_min'] - sample_df['time_min']) < BOUNDARY_TRIM_MIN)

sample_df['delta_soh'] = sample_df.groupby(['plane_id', 'battery_id', 'flight_id'])['observed_soh_pct'].diff().abs()
sample_df['delta_soc'] = sample_df.groupby(['plane_id', 'battery_id', 'flight_id'])['soc_pct'].diff().abs()
sample_df['delta_voltage'] = sample_df.groupby(['plane_id', 'battery_id', 'flight_id'])['voltage_v'].diff().abs()
sample_df['delta_temp'] = sample_df.groupby(['plane_id', 'battery_id', 'flight_id'])['temp_c'].diff().abs()

sample_df['soh_jump_mask'] = sample_df['delta_soh'] > SOH_DELTA_THRESHOLD
sample_df['soc_jump_mask'] = sample_df['delta_soc'] > SOC_DELTA_THRESHOLD
sample_df['volt_jump_mask'] = sample_df['delta_voltage'] > VOLT_DELTA_THRESHOLD
sample_df['temp_jump_mask'] = sample_df['delta_temp'] > TEMP_JUMP_THRESHOLD
sample_df['cell_spread_mask'] = sample_df['cell_v_spread'] > CELL_SPREAD_ABS_THRESHOLD

for metric in ['voltage_v', 'soc_pct', 'temp_c', 'cell_v_spread']:
    sample_df[f'{metric}_rz'] = sample_df.groupby(['plane_id', 'battery_id'])[metric].transform(robust_zscore)

sample_df['robust_outlier_mask'] = (
    sample_df['voltage_v_rz'].abs().gt(5)
    | sample_df['soc_pct_rz'].abs().gt(5)
    | sample_df['temp_c_rz'].abs().gt(5)
    | sample_df['cell_v_spread_rz'].abs().gt(5)
)

sample_df['invalid_mask'] = (
    sample_df['observed_soh_pct'].lt(0)
    | sample_df['observed_soh_pct'].gt(100)
    | sample_df['soc_pct'].lt(0)
    | sample_df['soc_pct'].gt(100)
    | sample_df['voltage_v'].lt(250)
    | sample_df['voltage_v'].gt(450)
    | sample_df['cell_v_spread'].lt(0)
)

mask_cols = [
    'boundary_mask',
    'soh_jump_mask',
    'soc_jump_mask',
    'volt_jump_mask',
    'temp_jump_mask',
    'cell_spread_mask',
    'robust_outlier_mask',
    'invalid_mask',
]
sample_df['exclude_mask'] = sample_df[mask_cols].any(axis=1)

sample_df[mask_cols + ['exclude_mask']].mean().sort_values(ascending=False)


In [ ]:
mask_summary = (
    sample_df.groupby(['plane_id', 'battery_id'], as_index=False)
    .agg(
        rows=('observed_soh_pct', 'size'),
        excluded_rows=('exclude_mask', 'sum'),
        excluded_frac=('exclude_mask', 'mean'),
        boundary_frac=('boundary_mask', 'mean'),
        soh_jump_frac=('soh_jump_mask', 'mean'),
        soc_jump_frac=('soc_jump_mask', 'mean'),
        volt_jump_frac=('volt_jump_mask', 'mean'),
        temp_jump_frac=('temp_jump_mask', 'mean'),
        cell_spread_frac=('cell_spread_mask', 'mean'),
        robust_outlier_frac=('robust_outlier_mask', 'mean'),
        invalid_frac=('invalid_mask', 'mean'),
    )
)
mask_summary


In [ ]:
raw_event = (
    sample_df.groupby(['plane_id', 'battery_id', 'flight_id', 'event_datetime'], as_index=False)
    .agg(raw_soh_pct=('observed_soh_pct', 'median'))
)

clean_event = (
    sample_df.loc[~sample_df['exclude_mask']]
    .groupby(['plane_id', 'battery_id', 'flight_id', 'event_datetime'], as_index=False)
    .agg(clean_soh_pct=('observed_soh_pct', 'median'))
)

event_compare = raw_event.merge(clean_event, on=['plane_id', 'battery_id', 'flight_id', 'event_datetime'], how='left')
event_compare.head()


In [ ]:
effect_summary = (
    event_compare.groupby(['plane_id', 'battery_id'], as_index=False)
    .agg(
        n_events=('raw_soh_pct', 'size'),
        cleaned_events=('clean_soh_pct', lambda s: s.notna().sum()),
        raw_std=('raw_soh_pct', 'std'),
        clean_std=('clean_soh_pct', 'std'),
        raw_range=('raw_soh_pct', lambda s: pd.to_numeric(s, errors='coerce').max() - pd.to_numeric(s, errors='coerce').min()),
        clean_range=('clean_soh_pct', lambda s: pd.to_numeric(s, errors='coerce').max() - pd.to_numeric(s, errors='coerce').min()),
    )
)
effect_summary['std_reduction_pct'] = 100.0 * (1.0 - effect_summary['clean_std'] / effect_summary['raw_std'])
effect_summary['range_reduction_pct'] = 100.0 * (1.0 - effect_summary['clean_range'] / effect_summary['raw_range'])
effect_summary


In [ ]:
for plane_id in PLANES:
    plane = event_compare.loc[event_compare['plane_id'] == plane_id].copy()
    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True, constrained_layout=True)
    for ax, battery_id in zip(axes, sorted(plane['battery_id'].unique())):
        g = plane.loc[plane['battery_id'] == battery_id].sort_values('event_datetime')
        ax.plot(g['event_datetime'], g['raw_soh_pct'], color='#9ecae1', linewidth=1.4, alpha=0.75, label='Raw event SOH')
        ax.plot(g['event_datetime'], g['clean_soh_pct'], color='#08519c', linewidth=2.0, label='Boundary/outlier masked SOH')
        ax.set_title(f'Plane {plane_id} Battery {battery_id}: raw vs masked event-level SOH')
        ax.set_ylabel('SOH (%)')
        ax.legend(loc='best')
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    axes[-1].set_xlabel('Event datetime')
    plt.show()


## How To Read This

- If `effect_summary` shows meaningful reduction in standard deviation and range, boundary or outlier masking is helping.
- If masking removes very few rows and barely changes event-level SOH, then the spikes are probably not just transient sample-level artifacts.
- If masking removes many rows and the series smooths substantially, then the raw observed SOH is likely being distorted by unstable boundary periods or bad telemetry states.

This notebook is intentionally conservative. If it only changes the series a little, the next step would be a more targeted regime mask or spike-date analysis rather than broader regression.